# Urdu OCR — Quickstart (run everything in one go)

**Before running anything: `Runtime > Change runtime type > T4 GPU`.**

This single notebook clones the repo, installs dependencies, generates a
synthetic starter dataset, fine-tunes TrOCR on it, evaluates it (prints
real CER/WER/accuracy), and launches a live demo — in that order, one
cell at a time. Run cells top to bottom.

Every cell is written to be **safe to re-run** (idempotent `cd`, no nested
clones) if your session disconnects partway through — just re-run from
Cell 1.


In [ ]:
# Cell 1 — Clone the repo (safe to re-run) and install dependencies.
import os

REPO_DIR = '/content/repo'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists, skipping clone.')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

!pip install -q -r requirements.txt
# Belt-and-suspenders: some transformers versions need these explicitly
# to load TrOCR's tokenizer even after requirements.txt is installed.
!pip install -q sentencepiece protobuf

import sys
if 'src' not in sys.path:
    sys.path.insert(0, 'src')

print('Setup complete.')


In [ ]:
# Cell 2 — Confirm GPU is attached. If this prints False, go to
# Runtime > Change runtime type > T4 GPU, then Runtime > Restart session,
# and re-run from Cell 1.
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Training will be extremely slow on CPU.')


## Generate a starter dataset

The repo ships the Urdu sentence corpus and fonts but not `data/labels.csv`
itself (it's gitignored, since it's generated, not hand-written). This
renders 400 synthetic images across 3 Urdu fonts/varied backgrounds — see
`docs/ACCURACY_NOTES.md` for why. Bump `num_images` up (or add your own
real photographed images to `data/raw/` + `data/labels_real.csv`, then
merge with `synthetic.merge_labels_csv`) for a stronger model once this
first run works end to end.


In [ ]:
# Cell 3 — Generate synthetic training data.
from synthetic import generate_dataset

os.makedirs('data', exist_ok=True)
stats = generate_dataset(
    output_dir='data/raw/synthetic',
    labels_csv='data/labels.csv',
    num_images=400,
    pseudo_fraction=0.3,
    seed=42,
)
print(f'Generated {stats.total} images ({stats.from_corpus} corpus + {stats.from_pseudo} pseudo)')
print('Fonts used:', stats.fonts_used)


## Train

~15-30 minutes on a free-tier T4 for 400 images / 6 epochs. Watch for
"Epoch ... eval_cer: ..." lines in the output — CER should trend down.
If this cell's output ends with a traceback instead of "Best model saved
to: ...", **stop and paste me the error** rather than continuing to the
next cell — evaluation will fail with a confusing, unrelated-looking error
if training didn't actually finish.


In [ ]:
# Cell 4 — Fine-tune TrOCR on the generated dataset.
!python src/train.py \
    --labels-csv data/labels.csv \
    --output-dir models/urdu-trocr \
    --epochs 6 --batch-size 8

import os
assert os.path.isdir('models/urdu-trocr/final'), (
    'Training did not produce models/urdu-trocr/final -- scroll up for the '
    'real error and do not continue to Cell 5 yet.'
)
print('Training finished, model saved to models/urdu-trocr/final')


## Evaluate

This prints your actual CER / WER / exact-match accuracy, plus the 5 worst predictions — this is the real number for your README.

In [ ]:
# Cell 5 — Evaluate on the held-out test set.
!python src/evaluate.py models/urdu-trocr/final \
    --test-csv data/test.csv \
    --report-csv reports/eval_report.csv


In [ ]:
# Cell 6 — (optional) preview the full per-sample report as a table.
import pandas as pd
report = pd.read_csv('reports/eval_report.csv')
report.sort_values('cer', ascending=False).head(10)


## Try it live

Launches a Gradio interface right here in Colab with a public share link — upload one of your own Urdu images and see the prediction. Screenshot this for your README (Week 5 handout's 'screenshot your working demo').

In [ ]:
# Cell 7 — Launch a live demo (Colab-hosted, public share link).
os.environ['MODEL_PATH'] = os.path.abspath('models/urdu-trocr/final')
os.environ['GRADIO_SHARE'] = 'true'

import importlib
import app
importlib.reload(app)
app.interface.launch(share=True)


## Next steps

- **Not accurate enough yet?** The single highest-leverage next step is
  more *real* photographed/downloaded images mixed into `data/labels.csv`
  (synthetic-only data plateaus — see `docs/ACCURACY_NOTES.md`). Re-run
  from Cell 3 with a bigger `num_images`, or add real data first.
- **Publish the model** so `app.py` can load it via `MODEL_PATH` without
  Colab open — see `notebooks/SI26_Week4_Train_Evaluate.ipynb`'s
  "Save your model" section for the `push_to_hub` snippet.
- **Deploy permanently** on HuggingFace Spaces — see
  `notebooks/SI26_Week5_Gradio_Deploy.ipynb` or the README's Live Demo
  section.
